# V3 × V3_1 — paired cross-evaluation (30 circles, no training)

**Purpose:** evaluate the **existing held-out V3 M2 and V3_1 M2 checkpoints** against **both** original v1 and revised v2 annotations on identical valid UAV pixels and spatial folds.

- **No fitting, checkpoint modification or data copying.** Uses the 3 spatial-fold M2 checkpoints for each generation, **never** the final all-30 production model.
- Each saved checkpoint runs once over each held-out window; both reference rasters are scored from the same predicted probabilities.
- Computes pixel TP/FP/FN/TN at threshold 0.5, pooled F1, precision, recall, IoU, circle cover MAE/RMSE/bias/R², per-site results, and model-versus-model paired deltas.
- Creates resumable per-circle CSV files in `Outputs/V3_1/02F_cross_reference_evaluation`; existing V3 and V3_1 files untouched.
- Reference geometry: `make_valid` of v2 row 519 is explicitly permitted per manifest; all 605 v2 rows are retained, with annotation contributions clipped by each evaluation circle. Non-overlapping six polygons therefore contribute nothing.

**Run top-to-bottom on a GPU (T4/L4).** The full evaluation may take a while because it reads all original validation windows.


In [ ]:
%pip -q install rasterio geopandas pyogrio shapely pandas matplotlib scikit-learn segmentation-models-pytorch
from pathlib import Path
import json, hashlib, gc
import numpy as np, pandas as pd, geopandas as gpd, rasterio, matplotlib.pyplot as plt
import torch, torch.nn as nn
from shapely.geometry import box
from shapely.ops import unary_union
from shapely import make_valid
from rasterio.windows import Window
from rasterio.features import rasterize
from sklearn.metrics import r2_score
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
ROOT=next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
                         Path('/content/drive/MyDrive/Musanga/Final')] if p.is_dir()),None)
assert ROOT is not None,'Musanga/Final missing'
BASE=ROOT/'Outputs/V3/02B_spatial3fold_no_MOS34'
NEW=ROOT/'Outputs/V3_1/02B_spatial3fold_no_MOS34'
REF0=ROOT/'Outputs/V3/01_reference_data_no_MOS34'
REF1=ROOT/'Outputs/V3_1/01_reference_data_no_MOS34'
OUT=ROOT/'Outputs/V3_1/02F_cross_reference_evaluation'
OUT.mkdir(parents=True,exist_ok=True)
A0=json.loads((ROOT/'Outputs/V3/00_input_audit/00_input_manifest.json').read_text())
A1=json.loads((ROOT/'Outputs/V3_1/00_input_audit/00_input_manifest.json').read_text())
assert A0['schema_version']=='v3_input_audit_5_nadine_circles'
assert A1['schema_version']=='v3_1_input_audit_nadine_v2'
assert A1['reference_version']=='Nadine_v2'
links=pd.read_csv(REF1/'01_circle_spatial_folds.csv')
oldlinks=pd.read_csv(REF0/'01_circle_spatial_folds.csv')
cols=['circle_id','site','mosaic','fold']
assert links[cols].sort_values('circle_id').reset_index(drop=True).equals(
    oldlinks[cols].sort_values('circle_id').reset_index(drop=True)), 'V3 spatial folds do not match V3_1'
assert len(links)==30 and links.circle_id.is_unique
windows=pd.read_csv(REF1/'01_candidate_windows.csv')
val=windows[windows.kind=='exhaustive'].copy()
assert set(val.circle_id.astype(int))>=set(links.circle_id.astype(int))
assert torch.cuda.is_available(),'Select Colab GPU Runtime (T4 or L4)'
device='cuda'
print('GPU:',torch.cuda.get_device_name(0))
print('Same 30 circle IDs/folds in V3 and V3_1: VERIFIED')

In [ ]:
def polygon_parts(g):
    if g.geom_type in ('Polygon','MultiPolygon'):
        return [g]
    if hasattr(g,'geoms'):
        return [p for item in g.geoms for p in polygon_parts(item)]
    return []

def read_clean(path,allowed_repair=None):
    raw=gpd.read_file(path)
    assert raw.crs is not None,str(path)
    geometries=[];inds=[]
    for i,g in enumerate(raw.geometry):
        if g is None or g.is_empty:continue
        if not g.is_valid:
            fixed=unary_union(polygon_parts(make_valid(g)))
            change=abs(fixed.area-g.area)/max(fixed.area,g.area,1e-12)
            allowed=(allowed_repair is not None and i in allowed_repair and
                     abs(change-allowed_repair[i])<.0005)
            limit=.01
            assert fixed.is_valid and not fixed.is_empty and (change<=limit or allowed), (
                'Unapproved polygon repair',path,i,change)
            g=fixed
        inds.append(i);geometries.append(g)
    result=raw.iloc[inds].copy().reset_index(drop=True)
    result.geometry=geometries
    result['original_row']=inds
    return result

repairs={int(x['source_row']):float(x['relative_area_change'])
         for x in A1['annotation_versioning'].get('provisionally_accepted_geometry_repairs',[])
         if x['version']=='v2'}
assert set(repairs)=={519},'Expected precisely audited v2 source row 519'
CIRCLE_FILE=Path(A1['exhaustive_annotations']['complete_roi_boundaries'])
V1_FILE=Path(A0['exhaustive_annotations']['positive_polygons'])
V2_FILE=Path(A1['exhaustive_annotations']['positive_polygons'])
circles=read_clean(CIRCLE_FILE)
v1=read_clean(V1_FILE)
v2=read_clean(V2_FILE,repairs)
assert len(circles)==34 and len(v1)==1394 and len(v2)==605
circles=circles[circles.original_row.isin(links.circle_id)].copy()
roi=read_clean(Path(A1['roi']))
ex_mosaic='Mos_3_4_transparent_mosaic_group1_georef.tif'
mosaics={m['file']:m for m in A1['mosaics'] if m['file']!=ex_mosaic}
geo={}
for name,m in mosaics.items():
    with rasterio.open(m['path']) as raster:
        geo[name]=dict(site=m['site'],path=m['path'],crs=raster.crs,
                       transform=raster.transform,
                       roi_local=roi.to_crs(raster.crs),
                       v1_local=v1.to_crs(raster.crs),
                       v2_local=v2.to_crs(raster.crs),
                       circles_local=circles.to_crs(raster.crs))
print('Reference polygons loaded:',len(v1),'(v1),',len(v2),'(v2)')
print('V2 approved geometry repairs:',repairs)

In [ ]:
def burn_window(geometries,name,win):
    """Rasterize geometries intersecting a crop, using the spatial index.

    Accept GeoSeries OR GeoDataFrame; always iterate actual Shapely geometries,
    never GeoDataFrame column names.
    """
    d=geo[name]
    series=geometries.geometry if isinstance(geometries,gpd.GeoDataFrame) else geometries
    assert isinstance(series,gpd.GeoSeries),type(series)
    bb=box(*rasterio.windows.bounds(win,d['transform']))
    ids=series.sindex.query(bb,predicate='intersects')
    shape=(int(win.height),int(win.width))
    if len(ids)==0:return np.zeros(shape,dtype=bool)
    selected=series.iloc[ids]
    return rasterize(((g,1) for g in selected.values
                      if g is not None and not g.is_empty),
        out_shape=shape,transform=rasterio.windows.transform(win,d['transform']),
        fill=0,dtype='uint8').astype(bool)

_ACTIVE_RASTER=None
_ACTIVE_NAME=None

def rgb_for_window(name,win):
    if _ACTIVE_RASTER is not None and _ACTIVE_NAME==name:
        raw=_ACTIVE_RASTER.read([1,2,3],window=win,masked=True)
    else:
        with rasterio.open(geo[name]['path']) as ds:
            raw=ds.read([1,2,3],window=win,masked=True)
    valid=~np.any(np.ma.getmaskarray(raw),axis=0)
    a=raw.filled(0).astype('float32')
    valid &= np.any(a!=0,axis=0)&np.any(a!=255,axis=0)
    valid &= ~((a[0]==0)&(a[1]==0)&(a[2]==255))
    return np.clip(a/255.,0,1).transpose(1,2,0).astype('float32'),valid



In [ ]:
class ResNet18UNet(nn.Module):
    def __init__(self):
        super().__init__()
        import segmentation_models_pytorch as smp
        self.net=smp.Unet(encoder_name='resnet18',encoder_weights=None,
                          in_channels=3,classes=1)
    def forward(self,x):return self.net(x)[:,0]

def checkpoint(version,fold):
    root=BASE if version=='V3' else NEW
    directory=root/f'fold_{fold}'
    assert directory.exists(),directory
    # V3 historically nested some checkpoints inside another fold directory.
    candidates=list(directory.glob('M2_resnet18_spatialCV_*.pt'))
    candidates+=list((directory/f'fold_{fold}').glob('M2_resnet18_spatialCV_*.pt'))
    candidates=list(sorted(set(candidates)))
    assert len(candidates)==1, (
        'Expected ONE version-specific M2 checkpoint; do not guess among files.',
        version,fold,[str(p) for p in candidates])
    return candidates[0]

manifest={}
for version in ['V3','V3_1']:
    for fold in [0,1,2]:
        p=checkpoint(version,fold)
        assert p.stat().st_size>1_000_000,(version,fold,p)
        manifest[f'{version}_{fold}']=dict(path=str(p),size=p.stat().st_size)
display(pd.DataFrame([dict(model=k,**v) for k,v in manifest.items()]))
(OUT/'02F_checkpoint_manifest.json').write_text(json.dumps(manifest,indent=2))
print('All six held-out M2 checkpoint paths resolved. No training will take place.')

In [ ]:
def evaluate_one(model,version,fold,circle):
    cid=int(circle.circle_id);name=str(circle.mosaic);d=geo[name]
    selection=val[(val.circle_id==cid)&(val.mosaic==name)]
    assert len(selection),('Missing validation windows',cid,name)
    cg=d['circles_local'].loc[d['circles_local'].original_row==cid].geometry
    counters={}
    for ref in ['v1','v2']:
        counters[ref]=dict(n=0,pos=0,sumprob=0.,tp=0,fp=0,fn=0,tn=0)
    for w in selection.itertuples(index=False):
        win=Window(int(w.col),int(w.row),int(w.width),int(w.height))
        mask=burn_window(cg,name,win)
        mask &= burn_window(d['roi_local'].geometry,name,win)
        rgb,valid=rgb_for_window(name,win)
        mask &= valid
        if not mask.any():continue
        truths={ref:burn_window(d[ref+'_local'].geometry,name,win)[mask]
                for ref in ['v1','v2']}
        x=torch.from_numpy(rgb.transpose(2,0,1).copy())[None].to(device)
        with torch.inference_mode():
            p=torch.sigmoid(model(x))[0].cpu().numpy()[mask]
        binary=p>=.5
        totalp=float(p.sum())
        for ref,truth in truths.items():
            z=counters[ref];n=len(truth);z['n']+=n
            z['pos']+=int(truth.sum());z['sumprob']+=totalp
            z['tp']+=int(np.sum(binary&truth))
            z['fp']+=int(np.sum(binary&~truth))
            z['fn']+=int(np.sum((~binary)&truth))
            z['tn']+=int(np.sum((~binary)&~truth))
    assert counters['v1']['n']==counters['v2']['n']>0,(version,cid,counters)
    rows=[]
    for ref,z in counters.items():
        tp,fp,fn=z['tp'],z['fp'],z['fn'];n=z['n']
        obs=100*z['pos']/n;pred=100*z['sumprob']/n
        rows.append(dict(model_version=version,reference_version=ref,fold=fold,
            site=circle.site,circle_id=cid,mosaic=name,valid_pixels=n,
            observed_fraction=obs,predicted_fraction=pred,
            bias_pp=pred-obs,abs_error_pp=abs(pred-obs),
            tp=tp,fp=fp,fn=fn,tn=z['tn'],
            pixel_f1=2*tp/max(1,2*tp+fp+fn),
            pixel_iou=tp/max(1,tp+fp+fn),
            pixel_precision=tp/max(1,tp+fp),
            pixel_recall=tp/max(1,tp+fn)))
    return rows

outrows=[]
for version in ['V3','V3_1']:
    for fold in [0,1,2]:
        model=ResNet18UNet().to(device)
        ckpt=checkpoint(version,fold)
        state=torch.load(ckpt,map_location=device,weights_only=True)
        model.load_state_dict(state,strict=True)
        model.eval()
        sub=links[links.fold==fold].sort_values(['site','circle_id'])
        print('====',version,'fold',fold,'held-out circles',len(sub),'====',flush=True)
        for circle in sub.itertuples(index=False):
            fn=OUT/f'circle_{version}_fold{fold}_id{int(circle.circle_id):02d}.csv'
            if fn.exists():
                previous=pd.read_csv(fn)
                assert set(previous.reference_version)=={'v1','v2'} and len(previous)==2
                rows=previous.to_dict('records')
            else:
                rows=evaluate_one(model,version,fold,circle)
                pd.DataFrame(rows).to_csv(fn,index=False)
            outrows.extend(rows)
            print(' ',circle.site,'circle',circle.circle_id,
                  'v1/v2 cover:',round(rows[0]['observed_fraction'],2),
                  round(rows[1]['observed_fraction'],2),
                  'pred:',round(rows[0]['predicted_fraction'],2),flush=True)
        del model,state;torch.cuda.empty_cache();gc.collect()
result=pd.DataFrame(outrows)
assert len(result)==120 and not result.duplicated(
    ['model_version','reference_version','circle_id']).any()
result.to_csv(OUT/'02F_paired_120_circle_scores.csv',index=False)
print('SAVED',OUT/'02F_paired_120_circle_scores.csv'# Exact checkpoints documented in the completed V3 and V3_1 NB02B runs.
# Multiple historical V3 checkpoints may coexist: never choose by glob order.
PINNED={
    'V3':{
        0:'M2_resnet18_spatialCV_ded9fba8097e31b4c6f0.pt',
        1:'M2_resnet18_spatialCV_c0e1469259692ab6a3fb.pt',
        2:'M2_resnet18_spatialCV_3f7145217cd8e1bd82d3.pt'},
    'V3_1':{
        0:'M2_resnet18_spatialCV_3053bd67a31dfd9ad367.pt',
        1:'M2_resnet18_spatialCV_d4dd0de428c5ba807a86.pt',
        2:'M2_resnet18_spatialCV_349d159fa28f10d1c475.pt'}
}
def checkpoint(version,fold):
    root=BASE if version=='V3' else NEW
    p=root/f'fold_{fold}'/PINNED[version][fold]
    assert p.exists(),('Published benchmark checkpoint is missing',version,fold,p)
    return p
)

In [ ]:
def summarize(g):
    tp,fp,fn=[int(g[c].sum()) for c in ['tp','fp','fn']]
    obs=g.observed_fraction.to_numpy(float)
    pred=g.predicted_fraction.to_numpy(float)
    residual=pred-obs
    return pd.Series(dict(circles=len(g),pixel_f1=2*tp/max(1,2*tp+fp+fn),
        pixel_iou=tp/max(1,tp+fp+fn),
        pixel_precision=tp/max(1,tp+fp),
        pixel_recall=tp/max(1,tp+fn),
        mean_observed_cover_pct=obs.mean(),mean_predicted_cover_pct=pred.mean(),
        fraction_MAE_pp=np.mean(np.abs(residual)),
        fraction_RMSE_pp=np.sqrt(np.mean(residual**2)),
        fraction_bias_pp=residual.mean(),
        fraction_R2=r2_score(obs,pred) if len(g)>1 and np.var(obs)>0 else np.nan))
overall=result.groupby(['model_version','reference_version'],sort=False).apply(
    summarize,include_groups=False).reset_index()
by_site=result.groupby(['site','model_version','reference_version'],sort=False).apply(
    summarize,include_groups=False).reset_index()
display(overall.round(3))
display(by_site.round(3))
overall.to_csv(OUT/'02F_2x2_overall_summary.csv',index=False)
by_site.to_csv(OUT/'02F_2x2_by_site_summary.csv',index=False)

# Each cell's model effect versus the SAME reference: meaningful paired contrasts.
pivot=result.pivot(index=['circle_id','site','fold','reference_version'],
                   columns='model_version',values='abs_error_pp').reset_index()
pivot['improvement_v31_pp']=pivot['V3']-pivot['V3_1']
paired=pivot.groupby(['site','reference_version'],sort=False).agg(
    n=('circle_id','size'),mean_MAE_improvement_pp=('improvement_v31_pp','mean'),
    improved_circles=('improvement_v31_pp',lambda x:int((x>0).sum()))).reset_index()
display(paired.round(3))
paired.to_csv(OUT/'02F_paired_model_improvements.csv',index=False)
print('2 × 2 cross-evaluation complete. Source checkpoints and previous results unchanged.')

## Benchmark consistency checks

Read the original V3 and V3_1 paired outputs. Compare original on-diagonal pixel counts and fractions to the rerun. Differences in observed or predicted fraction beyond numerical tolerance indicate a mismatched checkpoint, window selection, imagery, or reference rasterization; **do not interpret cross-evaluation until resolved.**


In [ ]:
def read_saved(version):
    root=BASE if version=='V3' else NEW
    csv=root/'02B_spatialCV_all30_paired.csv'
    assert csv.exists(),csv
    z=pd.read_csv(csv)
    z=z[z.model=='M2'].copy()
    assert z.circle_id.nunique()==30,len(z)
    return z[['circle_id','observed_fraction','predicted_fraction','valid_pixels']]

checks=[]
for version,ref in [('V3','v1'),('V3_1','v2')]:
    fresh=result[(result.model_version==version)&(result.reference_version==ref)].copy()
    old=read_saved(version)
    joined=fresh.merge(old,on='circle_id',suffixes=('_rerun','_saved'),validate='one_to_one')
    for item in ['observed_fraction','predicted_fraction']:
        diff=(joined[item+'_rerun']-joined[item+'_saved']).abs()
        maxdiff=float(diff.max())
        checks.append(dict(model=version,metric=item,max_absolute_difference_pp=maxdiff))
        assert maxdiff<0.15,(version,item,maxdiff,'Checkpoint/reference does not reproduce benchmark')
    assert (joined.valid_pixels_rerun==joined.valid_pixels_saved).all(), (
        version,'Valid pixel count changed: cannot compare fairly')
display(pd.DataFrame(checks).round(6))
pd.DataFrame(checks).to_csv(OUT/'02F_on_diagonal_reproduction_QA.csv',index=False)
print('PASS: Both published diagonals reproduce within 0.15 pp on all 30 circles.')